# SafeLens 交互可视化专项展示：Qwen3-0.6B

这个 notebook 专门展示 SafeLens 的交互式机制解释可视化。它会加载真实 `Qwen/Qwen3-0.6B` 权重，缓存真实 attention pattern、residual stream activation 和 logits，然后逐个展示交互部件。

每个可视化部件前都包含说明：
- 这个部件展示的是哪一种模型内部对象；
- 行、列、颜色、数值分别是什么意思；
- 可以怎么交互；
- 结合本 notebook 的样例输出应该如何解读。

注意术语：这里的 `d0, d1, ...` 表示 residual stream 的隐藏维度编号，不是 SAE 语义特征，也不是训练数据 feature。

## 0. 环境准备

这个 cell 会优先使用当前仓库的 `src/`。如果缺少必要依赖，会从本地 checkout 做 editable install。

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

SAFELENS_INSTALL_EXTRA = "models"
FORCE_EDITABLE_INSTALL = False


def _find_safelens_checkout(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "SafeLens").is_dir():
            return candidate
    return None


def _module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


PROJECT_ROOT = _find_safelens_checkout(Path.cwd().resolve())
if PROJECT_ROOT is not None:
    src_dir = PROJECT_ROOT / "src"
    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

core_modules = ["pydantic", "yaml"]
extra_modules = {"models": ["torch", "transformers"], "viz": ["circuitsvis"]}
requested_extras = [extra.strip() for extra in SAFELENS_INSTALL_EXTRA.split(",") if extra.strip()]
missing_modules = [
    name
    for name in ["SafeLens", *core_modules, *[m for extra in requested_extras for m in extra_modules[extra]]]
    if _module_missing(name)
]

if FORCE_EDITABLE_INSTALL or missing_modules:
    if PROJECT_ROOT is None:
        raise RuntimeError("SafeLens is not importable and no local checkout was found.")
    install_target = str(PROJECT_ROOT)
    if requested_extras:
        install_target = f"{install_target}[{','.join(requested_extras)}]"
    cmd = [sys.executable, "-m", "pip", "install", "-e", install_target, "--no-build-isolation"]
    print("Installing SafeLens because these modules are missing:", missing_modules)
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    importlib.invalidate_caches()
else:
    print("SafeLens import path and requested dependencies are ready.")

print("SafeLens project root:", PROJECT_ROOT or "not found")

SafeLens import path and requested dependencies are ready.
SafeLens project root: /workspace/SafeLens


## 1. 加载真实模型并缓存内部激活

这个 cell 使用真实 `Qwen/Qwen3-0.6B` 权重。为了 notebook 运行速度和显存可控，只缓存：
- `layer_0/1/2.pattern`：attention pattern，形状是 `[batch, head, dest_token, source_token]`；
- `layer_0/1/2.resid_post`：每层 MLP 后的 residual stream，形状是 `[batch, token, d_model]`；
- `logits`：用于 next-token prediction browser。

本样例 prompt 是：`SafeLens interactive views reveal attention heads, residual dimensions, and model predictions.`

In [2]:
from __future__ import annotations

from pathlib import Path

import torch
from IPython.display import HTML, Markdown, display

from SafeLens import (
    ModelLoadConfig,
    plot_activation_cache_browser,
    plot_activation_patching_browser,
    plot_attention_browser,
    plot_attention_heads,
    plot_mlp_component_browser,
    plot_mlp_output_direction_viewer,
    plot_mlp_neuron_topk_browser,
    plot_mlp_logit_contribution_browser,
    plot_next_token_browser,
    plot_text_neuron_browser,
    plot_topk_samples_browser,
    plot_topk_tokens_browser,
)
from SafeLens.utils import build_model_wrapper

MODEL_ID = "Qwen/Qwen3-0.6B"
CACHE_DIR = Path("../.cache/safelens/qwen3-0.6b-interactive").resolve()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "bfloat16" if DEVICE == "cuda" else "float32"
PROMPT = "SafeLens interactive views reveal attention heads, residual dimensions, and model predictions."

qwen_config = ModelLoadConfig(
    source="qwen3_dense",
    name=MODEL_ID,
    dtype=DTYPE,
    device=DEVICE,
    cache_dir=str(CACHE_DIR),
    trust_remote_code=True,
)
qwen_wrapper = build_model_wrapper(qwen_config)
qwen_model = qwen_wrapper.load_model()

viz_token_tensor = qwen_wrapper.to_tokens(PROMPT, prepend_bos=True)
viz_tokens = qwen_wrapper.to_str_tokens(viz_token_tensor)
qwen_logits, qwen_cache = qwen_wrapper.run_with_cache(
    viz_token_tensor,
    layers=(
        "layer_0.pattern",
        "layer_1.pattern",
        "layer_2.pattern",
        "layer_0.resid_post",
        "layer_1.resid_post",
        "layer_2.resid_post",
        "layer_0.mlp_out",
        "layer_1.mlp_out",
        "layer_2.mlp_out",
        "layer_0.post",
        "layer_1.post",
        "layer_2.post",
    ),
    return_cache_object=True,
)

print("model:", MODEL_ID)
print("device:", DEVICE)
print("prompt tokens:", viz_tokens)
print("token count:", len(viz_tokens))
print("layer_0.pattern shape:", tuple(qwen_cache["pattern", 0].shape))
print("layer_0.resid_post shape:", tuple(qwen_cache["resid_post", 0].shape))
print("layer_0.mlp_out shape:", tuple(qwen_cache["mlp_out", 0].shape))
print("layer_0.post shape:", tuple(qwen_cache["post", 0].shape))

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 18111.78it/s]

model: Qwen/Qwen3-0.6B
device: cuda
prompt tokens: ['Safe', 'Lens', ' interactive', ' views', ' reveal', ' attention', ' heads', ',', ' residual', ' dimensions', ',', ' and', ' model', ' predictions', '.']
token count: 15
layer_0.pattern shape: (1, 16, 15, 15)
layer_0.resid_post shape: (1, 15, 1024)
layer_0.mlp_out shape: (1, 15, 1024)
layer_0.post shape: (1, 15, 3072)


## 2. 构造交互可视化对象

这个 cell 只做数据整理，不直接展示部件。关键约定如下：
- `L0H0` 表示 layer 0 的 attention head 0；
- `dest_token` 是当前被更新的位置，`source_token` 是它读取信息的位置；
- `resid_d0` 到 `resid_d7` 是 residual stream 的前 8 个隐藏维度；
- cache browser 里展示的是前 16 个 residual dimensions 的 robust z-score，避免 raw residual 的极端值把色阶拉爆。

这里的 `Feature-Linked Text Sample Browser` 使用一个小型 demo text corpus 来展示“按 feature/层/样本筛选文本片段”的交互形式；它不是声称找到了 Qwen 的真实训练样本。

In [3]:
def _matrix_to_float_list(matrix):
    if hasattr(matrix, "detach"):
        return matrix.detach().float().cpu().tolist()
    return [[float(value) for value in row] for row in matrix]


def _markdown_table(headers, rows):
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = [
        "| " + " | ".join(str(row.get(header, "")).replace("|", "\\|") for header in headers) + " |"
        for row in rows
    ]
    return "\n".join([header, separator, *body])


layers = [0, 1, 2]
head_count = 4
MLP_TOPK_MAX = 8
MLP_DIRECTION_NEURON_COUNT = 8
MLP_DIRECTION_ITEM_COUNT = 8
layer_head_patterns = torch.stack(
    [qwen_cache["pattern", layer][0, :head_count].float().cpu() for layer in layers],
    dim=0,
)
attention_heads = [
    _matrix_to_float_list(layer_head_patterns[0, 0]),
    _matrix_to_float_list(layer_head_patterns[0, 1]),
    _matrix_to_float_list(layer_head_patterns[1, 0]),
    _matrix_to_float_list(layer_head_patterns[2, 0]),
]
attention_head_names = ["L0H0", "L0H1", "L1H0", "L2H0"]

resid_by_layer = torch.stack(
    [qwen_cache["resid_post", layer][0].float().cpu() for layer in layers],
    dim=0,
)
resid_dim_slice = resid_by_layer[:, :, :8]
text_dimension_acts = resid_dim_slice.permute(1, 0, 2).contiguous()

mlp_out_by_layer = torch.stack(
    [qwen_cache["mlp_out", layer][0].float().cpu() for layer in layers],
    dim=0,
)
mlp_post_by_layer = torch.stack(
    [qwen_cache["post", layer][0].float().cpu() for layer in layers],
    dim=0,
)
mlp_out_dim_slice = mlp_out_by_layer[:, :, :8]
mlp_post_neuron_slice = mlp_post_by_layer[:, :, :8]
mlp_component_acts = {
    "mlp_out": mlp_out_dim_slice.permute(1, 0, 2).contiguous(),
    "post": mlp_post_neuron_slice.permute(1, 0, 2).contiguous(),
}
mlp_post_full_acts = mlp_post_by_layer.permute(1, 0, 2).contiguous()

l2_scores = resid_by_layer.norm(dim=-1)
mean_abs_scores = resid_by_layer.abs().mean(dim=-1)
max_abs_scores = resid_by_layer.abs().amax(dim=-1)
path_filter_values = torch.stack([l2_scores, mean_abs_scores, max_abs_scores], dim=0)

# Raw residual activations can contain large boundary-token outliers. This display
# keeps real Qwen values but robustly normalizes each dimension for readability.
activation_cache_display = {}
for layer in layers:
    raw = qwen_cache["resid_post", layer][0].float().cpu()[:, :16]
    centered = raw - raw.median(dim=0).values
    scale = centered.abs().median(dim=0).values.clamp_min(1e-3)
    robust_z = (centered / scale).clamp(-4.0, 4.0)
    activation_cache_display[f"layer_{layer}.resid_post_robust_z"] = robust_z
activation_cache_keys = list(activation_cache_display.keys())

resid_dim_labels_8 = [f"resid_d{i}" for i in range(text_dimension_acts.shape[-1])]
resid_dim_labels_16 = [f"resid_d{i}" for i in range(16)]
mlp_out_dim_labels_8 = [f"resid_d{i}" for i in range(mlp_out_dim_slice.shape[-1])]
mlp_neuron_labels_8 = [f"mlp_n{i}" for i in range(mlp_post_neuron_slice.shape[-1])]
mlp_all_neuron_labels = [f"mlp_n{i}" for i in range(mlp_post_full_acts.shape[-1])]
layer_labels = [f"L{layer}" for layer in layers]
head_labels = [f"H{i}" for i in range(head_count)]

mlp_strength_by_neuron = mlp_post_by_layer.abs().amax(dim=(0, 1))
mlp_direction_neuron_ids = torch.topk(
    mlp_strength_by_neuron,
    k=MLP_DIRECTION_NEURON_COUNT,
).indices.cpu()
mlp_direction_neuron_labels = [f"mlp_n{int(index)}" for index in mlp_direction_neuron_ids]


def _qwen_down_projection_weight(layer):
    return qwen_model.model.layers[layer].mlp.down_proj.weight.detach().float().cpu()


mlp_output_directions = torch.stack(
    [
        _qwen_down_projection_weight(layer)[:, mlp_direction_neuron_ids].T.contiguous()
        for layer in layers
    ],
    dim=0,
)
full_resid_dim_labels = [f"resid_d{i}" for i in range(mlp_output_directions.shape[-1])]

lm_head_weight = qwen_model.lm_head.weight.detach().float().cpu()
mlp_vocab_direction_scores = torch.matmul(mlp_output_directions, lm_head_weight.T)


def _top_vocab_entries(scores, *, largest=True, k=MLP_DIRECTION_ITEM_COUNT):
    values, indices = torch.topk(scores, k=k, largest=largest)
    return [
        {"label": qwen_wrapper.to_single_str_token(int(index)), "value": float(value)}
        for value, index in zip(values, indices, strict=True)
    ]


mlp_vocab_positive = [
    [_top_vocab_entries(neuron_scores, largest=True) for neuron_scores in layer_scores]
    for layer_scores in mlp_vocab_direction_scores
]
mlp_vocab_negative = [
    [_top_vocab_entries(neuron_scores, largest=False) for neuron_scores in layer_scores]
    for layer_scores in mlp_vocab_direction_scores
]

contribution_positions = list(range(len(viz_tokens) - 1))
actual_next_token_ids = viz_token_tensor[0, 1:].detach().cpu()
top_pred_token_ids = qwen_logits[0, :-1].detach().float().cpu().argmax(dim=-1)
mlp_output_for_contribution = mlp_out_by_layer[:, :-1, :]
actual_next_unembed = lm_head_weight[actual_next_token_ids]
top_pred_unembed = lm_head_weight[top_pred_token_ids]
actual_next_mlp_contrib = torch.einsum(
    "lpd,pd->lp",
    mlp_output_for_contribution,
    actual_next_unembed,
)
top_pred_mlp_contrib = torch.einsum(
    "lpd,pd->lp",
    mlp_output_for_contribution,
    top_pred_unembed,
)
mlp_logit_contributions = {
    "actual next token": actual_next_mlp_contrib,
    "model top token": top_pred_mlp_contrib,
}
def _compact_token_label(token: str) -> str:
    compact = token.strip().replace("\n", "\\n")
    return compact if compact else repr(token)


mlp_contribution_token_labels = [
    (
        f"{pos}:"
        f"{_compact_token_label(viz_tokens[pos])}"
        f"->{_compact_token_label(viz_tokens[pos + 1])}"
    )
    for pos in contribution_positions
]

sample_corpus = [
    ["training", "snippet", "safe", "answer"],
    ["unsafe", "request", "blocked", "policy"],
    ["attention", "copy", "residual", "trace"],
]
sample_tokens = [
    [sample_corpus, sample_corpus],
    [sample_corpus, sample_corpus],
]
sample_activations = [
    [
        [[0.05, 0.12, 0.38, 0.24], [0.08, 0.16, 0.92, 0.47], [0.22, 0.51, 0.77, 0.45]],
        [[0.18, 0.62, 0.31, 0.28], [0.14, 0.55, 0.49, 0.88], [0.19, 0.29, 0.83, 0.72]],
    ],
    [
        [[0.28, 0.24, 0.48, 0.61], [0.21, 0.18, 0.66, 0.73], [0.42, 0.81, 0.59, 0.52]],
        [[0.11, 0.35, 0.57, 0.46], [0.27, 0.41, 0.75, 0.64], [0.39, 0.86, 0.71, 0.67]],
    ],
]

interactive_views = {
    "01_attention_head_highlighter": plot_attention_heads(
        attention_heads,
        tokens=viz_tokens,
        head_names=attention_head_names,
        title="Interactive Attention Head Highlighter",
    ),
    "02_attention_path_browser": plot_attention_browser(
        layer_head_patterns,
        tokens=viz_tokens,
        layer_labels=layer_labels,
        head_labels=head_labels,
        title="Attention Path Browser",
    ),
    "03_residual_path_filter": plot_activation_patching_browser(
        path_filter_values,
        layers=layer_labels,
        positions=viz_tokens,
        slice_labels=["residual_l2", "residual_mean_abs", "residual_max_abs"],
        slice_axis_name="score",
        title="Residual Path Filter",
    ),
    "04_residual_dimension_browser": plot_text_neuron_browser(
        viz_tokens,
        text_dimension_acts,
        layer_labels=layer_labels,
        neuron_labels=resid_dim_labels_8,
        title="Token x Layer x Residual Dimension Browser",
    ),
    "05_mlp_component_browser": plot_mlp_component_browser(
        viz_tokens,
        mlp_component_acts,
        layer_labels=layer_labels,
        dimension_labels={
            "mlp_out": mlp_out_dim_labels_8,
            "post": mlp_neuron_labels_8,
        },
        title="MLP Component Browser",
    ),
    "06_mlp_neuron_topk_browser": plot_mlp_neuron_topk_browser(
        viz_tokens,
        mlp_post_full_acts,
        max_k=MLP_TOPK_MAX,
        layer_labels=layer_labels,
        neuron_labels=mlp_all_neuron_labels,
        title="MLP Neuron Top-K Browser",
    ),
    "07_mlp_output_direction_viewer": plot_mlp_output_direction_viewer(
        mlp_output_directions,
        layer_labels=layer_labels,
        neuron_labels=mlp_direction_neuron_labels,
        residual_labels=full_resid_dim_labels,
        vocab_positive=mlp_vocab_positive,
        vocab_negative=mlp_vocab_negative,
        max_items=MLP_DIRECTION_ITEM_COUNT,
        title="MLP Output Direction Viewer",
    ),
    "08_mlp_logit_contribution_browser": plot_mlp_logit_contribution_browser(
        mlp_logit_contributions,
        layer_labels=layer_labels,
        token_labels=mlp_contribution_token_labels,
        title="MLP Logit Contribution Browser",
    ),
    "09_topk_input_token_browser": plot_topk_tokens_browser(
        viz_tokens,
        text_dimension_acts.permute(1, 0, 2).contiguous(),
        max_k=4,
        layer_labels=layer_labels,
        neuron_labels=resid_dim_labels_8,
        title="Top-K Input Token by Residual Dimension Browser",
    ),
    "10_feature_linked_text_sample_browser": plot_topk_samples_browser(
        sample_tokens,
        sample_activations,
        layer_labels=["demo L0", "demo L1"],
        neuron_labels=["demo feature 0", "demo feature 1"],
        title="Feature-Linked Text Sample Browser",
    ),
    "11_next_token_prediction_browser": plot_next_token_browser(
        viz_token_tensor,
        qwen_logits,
        qwen_wrapper.to_single_str_token,
        top_k=8,
        title="Qwen Next-Token Prediction Browser",
    ),
    "12_residual_dimension_cache_browser": plot_activation_cache_browser(
        activation_cache_display,
        keys=activation_cache_keys,
        x_labels=resid_dim_labels_16,
        y_labels=viz_tokens,
        x_axis="Residual dimension",
        y_axis="Token",
        max_columns=16,
        title="Qwen Residual Dimension Cache Browser (robust z-score)",
    ),
}

component_coverage = [
    {
        "component": "Attention heads / attention patterns",
        "widgets": "01_attention_head_highlighter, 02_attention_path_browser",
        "evidence": "real Qwen layer_0/layer_1/layer_2 pattern cache",
    },
    {
        "component": "Residual stream dimensions / neuron-like activations",
        "widgets": "03_residual_path_filter, 04_residual_dimension_browser, 09_topk_input_token_browser, 12_residual_dimension_cache_browser",
        "evidence": "real Qwen residual stream activations from layers 0-2",
    },
    {
        "component": "MLP outputs / gated hidden activations",
        "widgets": "05_mlp_component_browser, 06_mlp_neuron_topk_browser, 07_mlp_output_direction_viewer, 08_mlp_logit_contribution_browser",
        "evidence": "real Qwen mlp_out and post hooks from layers 0-2",
    },
    {
        "component": "Feature-linked text samples",
        "widgets": "10_feature_linked_text_sample_browser",
        "evidence": "demo text sample corpus shaped like feature-linked training/text examples",
    },
    {
        "component": "Output logits / next-token predictions",
        "widgets": "11_next_token_prediction_browser",
        "evidence": "real Qwen logits at each prompt position",
    },
]
interaction_coverage = [
    {
        "operation": "Highlight or focus model features",
        "widgets": "01_attention_head_highlighter, 02_attention_path_browser",
        "controls": "hover/click head selector, hover/click token focus, hover/click heatmap cells",
    },
    {
        "operation": "Filter internal reasoning paths",
        "widgets": "02_attention_path_browser, 03_residual_path_filter, 12_residual_dimension_cache_browser",
        "controls": "layer/head/score/activation selectors and query-key transpose",
    },
    {
        "operation": "Inspect MLP components and neuron rankings",
        "widgets": "05_mlp_component_browser, 06_mlp_neuron_topk_browser",
        "controls": "component/layer/dimension selectors, token clicks, and top-k neuron ranking",
    },
    {
        "operation": "Trace MLP output directions and logit effects",
        "widgets": "07_mlp_output_direction_viewer, 08_mlp_logit_contribution_browser",
        "controls": "layer/neuron selectors, target selector, and click-to-read contribution cells",
    },
    {
        "operation": "Browse/search feature-linked examples",
        "widgets": "09_topk_input_token_browser, 10_feature_linked_text_sample_browser",
        "controls": "layer/neuron filters, top-bottom toggle, text query filter",
    },
    {
        "operation": "Inspect next-token behavior by position and metric",
        "widgets": "11_next_token_prediction_browser",
        "controls": "position selector and logit/log_prob/prob metric selector",
    },
]
assert len(component_coverage) >= 3
assert len(interaction_coverage) >= 3
assert len(interactive_views) >= 12

display(Markdown("### Neural Component Coverage"))
display(Markdown(_markdown_table(["component", "widgets", "evidence"], component_coverage)))
display(Markdown("### Interactive Operation Coverage"))
display(Markdown(_markdown_table(["operation", "widgets", "controls"], interaction_coverage)))

print("interactive widgets:", sorted(interactive_views))
print("residual dimensions displayed:", resid_dim_labels_16)
print("mlp output dimensions displayed:", mlp_out_dim_labels_8)
print("mlp hidden neurons displayed:", mlp_neuron_labels_8)
print("mlp direction neurons:", mlp_direction_neuron_labels)
print(
    "cache browser value range:",
    min(v.min().item() for v in activation_cache_display.values()),
    max(v.max().item() for v in activation_cache_display.values()),
)

### Neural Component Coverage

| component | widgets | evidence |
| --- | --- | --- |
| Attention heads / attention patterns | 01_attention_head_highlighter, 02_attention_path_browser | real Qwen layer_0/layer_1/layer_2 pattern cache |
| Residual stream dimensions / neuron-like activations | 03_residual_path_filter, 04_residual_dimension_browser, 09_topk_input_token_browser, 12_residual_dimension_cache_browser | real Qwen residual stream activations from layers 0-2 |
| MLP outputs / gated hidden activations | 05_mlp_component_browser, 06_mlp_neuron_topk_browser, 07_mlp_output_direction_viewer, 08_mlp_logit_contribution_browser | real Qwen mlp_out and post hooks from layers 0-2 |
| Feature-linked text samples | 10_feature_linked_text_sample_browser | demo text sample corpus shaped like feature-linked training/text examples |
| Output logits / next-token predictions | 11_next_token_prediction_browser | real Qwen logits at each prompt position |

### Interactive Operation Coverage

| operation | widgets | controls |
| --- | --- | --- |
| Highlight or focus model features | 01_attention_head_highlighter, 02_attention_path_browser | hover/click head selector, hover/click token focus, hover/click heatmap cells |
| Filter internal reasoning paths | 02_attention_path_browser, 03_residual_path_filter, 12_residual_dimension_cache_browser | layer/head/score/activation selectors and query-key transpose |
| Inspect MLP components and neuron rankings | 05_mlp_component_browser, 06_mlp_neuron_topk_browser | component/layer/dimension selectors, token clicks, and top-k neuron ranking |
| Trace MLP output directions and logit effects | 07_mlp_output_direction_viewer, 08_mlp_logit_contribution_browser | layer/neuron selectors, target selector, and click-to-read contribution cells |
| Browse/search feature-linked examples | 09_topk_input_token_browser, 10_feature_linked_text_sample_browser | layer/neuron filters, top-bottom toggle, text query filter |
| Inspect next-token behavior by position and metric | 11_next_token_prediction_browser | position selector and logit/log_prob/prob metric selector |

interactive widgets: ['01_attention_head_highlighter', '02_attention_path_browser', '03_residual_path_filter', '04_residual_dimension_browser', '05_mlp_component_browser', '06_mlp_neuron_topk_browser', '07_mlp_output_direction_viewer', '08_mlp_logit_contribution_browser', '09_topk_input_token_browser', '10_feature_linked_text_sample_browser', '11_next_token_prediction_browser', '12_residual_dimension_cache_browser']
residual dimensions displayed: ['resid_d0', 'resid_d1', 'resid_d2', 'resid_d3', 'resid_d4', 'resid_d5', 'resid_d6', 'resid_d7', 'resid_d8', 'resid_d9', 'resid_d10', 'resid_d11', 'resid_d12', 'resid_d13', 'resid_d14', 'resid_d15']
mlp output dimensions displayed: ['resid_d0', 'resid_d1', 'resid_d2', 'resid_d3', 'resid_d4', 'resid_d5', 'resid_d6', 'resid_d7']
mlp hidden neurons displayed: ['mlp_n0', 'mlp_n1', 'mlp_n2', 'mlp_n3', 'mlp_n4', 'mlp_n5', 'mlp_n6', 'mlp_n7']
mlp direction neurons: ['mlp_n55', 'mlp_n128', 'mlp_n1489', 'mlp_n321', 'mlp_n46', 'mlp_n646', 'mlp_n0', 'mlp

## 3.1 Interactive Attention Head Highlighter

**展示对象。** 真实 Qwen attention pattern。这里展示 `L0H0`, `L0H1`, `L1H0`, `L2H0` 四个 attention head。

**怎么看。** 大图的行是 destination token��也��是当前被更新的位置；列是 source token，也就是这个位置读取的信息来源。颜色越深表示 attention weight 越大。Overview 会把每个格子里最强的 head 用对应颜色显示出来。

**怎么交互。** 右侧 head selector 可以 hover 临时查看某个 head，click 锁定某个 head；下方 token strip 可以 hover/click 聚焦 token，查看它作为 destination 或 source 时的注意力分布。

**结合样例。** 如果某一行在前文 token 上有深色块，说明该 token 的表示正在从那个 source token 读取信息；如果对角线强，通常表示当前位置强烈关注自己或相邻上下文。

In [4]:
display(HTML(interactive_views['01_attention_head_highlighter'].html))

## 3.2 Attention Path Browser

**展示对象。** 多层多头 attention pattern，形状来自真实 cache：`[layer, head, dest_token, source_token]`。

**怎么看。** 下拉框里的 `L0 / H0` 等选择的是 layer 和 head。热图行仍是 destination token，列是 source token。

**怎么交互。** 你可以用 matrix/head 下拉框切换内部路径；`view` 下拉框可以切换 query-to-key / key-to-query 视角；hover/click 单元格会高亮对应行列。

**结合样例。** 它适合回答“某一层某个 head 在哪个 token 位置读取了哪段上下文”。这比只看单个 head 更适合比较不同 head 的路径。

In [5]:
display(HTML(interactive_views['02_attention_path_browser'].html))

## 3.3 Residual Path Filter

**展示对象。** 真实 Qwen residual stream 的 token-by-layer 摘要。这里不是做真实 activation patching，而是复用可交互 heatmap browser 来展示不同 residual score。

**怎么看。** 行是 layer，列是 token。`residual_l2` 是该 token residual 向量的 L2 norm；`residual_mean_abs` 是平均绝对值；`residual_max_abs` 是最大绝对值。

**怎么交互。** `score` 下拉框切换不同 residual 统计；hover/click 单元格会聚焦某个 layer-token 位置。

**结合样例。** 如果某个 token 在某层的 residual score 明显更高，它可能是这个 prompt 中表示变化较强的位置，值得进一步到 residual dimension browser 里看具体维度。

In [6]:
display(HTML(interactive_views['03_residual_path_filter'].html))

## 3.4 Token x Layer x Residual Dimension Browser

**展示对象。** 真实 Qwen `resid_post` 的前 8 个 residual stream dimensions。这里的 `resid_d0`、`resid_d1` 等只是隐藏维度编号，不是可解释语义 feature。

**怎么看。** token strip 中每个 token 的颜色表示当前选择的 layer 和 residual dimension 上的激活值。红/蓝表示相对正负方向，颜色深浅表示强度。

**怎么交互。** 下拉框切换 sample、layer、residual dimension；token 颜色会随选择实时变化。点击某个 token 后，下方 readout 会输出这个 token 在当前 layer 和 residual dimension 上的具体 activation value，包括 token index、token text 和数值。

**结合样例。** 例如选择 `L1` 和 `resid_d3` 后，哪个 token 颜色最深，就说明这个 residual dimension 在该 token 位置激活最强。点击该 token 可以看到类似 `token[7] "heads" = 1.2345678` 的读数；它能帮助定位“哪个维度在哪些 token 上活跃”，并把颜色判断落到可记录的具体数值上。

In [7]:
display(HTML(interactive_views['04_residual_dimension_browser'].html))

## 3.5 MLP Component Browser

**展示对象。** 真实 Qwen MLP hook：`mlp_out` 的前 8 个 residual-stream 输出维度，以及 `post` 的前 8 个 gated MLP hidden neurons。`mlp_out` 是 MLP 经过 down projection 后写回 residual stream 的向量；`post` 是送入 down projection 前的 gated hidden activation。`resid_d*` 和 `mlp_n*` 都只是维度编号，不是已经解释出的语义 feature。

**怎么看。** 当 component 选 `mlp_out` 时，token strip 的颜色表示当前 layer 上 MLP 输出在某个 residual dimension 的值。当 component 选 `post` 时，颜色表示某个 MLP hidden neuron 在各 token 位置的激活值。红/蓝表示相对正负方向，颜色深浅表示强度。

**怎么交互。** component 下拉框在 `mlp_out` 和 `post` 之间切换；layer 和 dimension 下拉框选择层和维度。点击任意 token 后，下方 readout 会输出该 token 在当前 MLP component/layer/dimension 上的具体值。

**结合样例。** 例如选择 `post`、`L1`、`mlp_n3` 后，颜色最深的 token 表示该 MLP hidden neuron 在该 token 位置响应最强。切到 `mlp_out` 后，同样的读数表示 MLP 最终写入 residual stream 某个维度的贡献。

In [8]:
display(HTML(interactive_views['05_mlp_component_browser'].html))

## 3.6 MLP Neuron Top-K Browser

**展示对象。** 真实 Qwen `post` MLP hidden activations。这里不再只看前几个 `mlp_n*`，而是在每个 token 和 layer 上从全部 MLP hidden neurons 中动态取 top-k。

**怎么看。** 上方 token strip 表示当前 layer 下每个 token 的 MLP neuron 激活摘要；下方排名列表显示当前 token 上最强的 MLP neurons。`rank by absolute` 看绝对值最大，`positive` 看最大正激活，`negative` 看最负激活。

**怎么交互。** 切换 layer/token/rank mode；也可以直接点击 token strip 中的 token。排名列表会实时更新，每一行的条形长度表示该 neuron 在当前 top-k 内的相对强度。

**结合样例。** 如果某个 token（比如 `attention`）在多个 layer 中都反复出现同一批高激活 MLP neurons，这些 neurons 值得进一步做输出方向或 logit contribution 分析。

In [9]:
display(HTML(interactive_views['06_mlp_neuron_topk_browser'].html))

## 3.7 MLP Output Direction Viewer

**展示对象。** 真实 Qwen MLP `down_proj` 的输出方向。这里自动选择在本 prompt 中 `post` 激活最强的若干 MLP neurons，并展示这些 neurons 的 `W_out` residual 写入方向，以及近似 unembedding 后最提升/压低的 vocab tokens。

**怎么看。** `Positive residual directions` 表示该 neuron 输出方向中最大的 residual dimensions；`Negative residual directions` 表示最负的 residual dimensions。`Promoted vocab tokens` 和 `Suppressed vocab tokens` 是用 output direction 直接投到 `lm_head` 得到的近似 vocab 方向，未经过最终 RMSNorm，因此应理解为方向性线索，不是完整 logits。

**怎么交互。** 切换 layer 和 neuron。四个列表同步更新；条形长度表示当前列表内相对强度。

**结合样例。** 如果某个高激活 `mlp_n` 同时强烈提升某些 token，说明它可能把 residual stream 推向这些词的方向。下一步可以结合 logit contribution heatmap 看它在具体位置是否真的影响预测。

In [10]:
display(HTML(interactive_views['07_mlp_output_direction_viewer'].html))

## 3.8 MLP Logit Contribution Browser

**展示对象。** 真实 Qwen `mlp_out` 对 next-token logit 的直接贡献近似值。每个格子是某层某 token 位置的 `mlp_out @ W_U[target]`，这里提供两个 target 视角：真实 prompt 的下一个 token，以及模型在该位置自己的 top prediction。

**怎么看。** 行是 layer，列是 token position。红色表示该层 MLP 输出把目标 token logit 往上推，蓝色表示往下压。颜色越深，直接贡献越大。

**怎么交互。** target 下拉框切换 `actual next token` 和 `model top token`；hover/click 热力图格子可以看到具体 layer-token contribution 数值。

**结合样例。** 如果某个 token 位置在 `actual next token` 视角为强红色，说明那一层 MLP 输出直接支持真实续写 token；如果在 `model top token` 视角更强，说明 MLP 更支持模型自己偏好的续写。

In [11]:
display(HTML(interactive_views['08_mlp_logit_contribution_browser'].html))

## 3.9 Top-K Input Token by Residual Dimension Browser

**展示对象。** 每个 layer 和 residual dimension 上，输入 token 的 top/bottom 激活排序。

**怎么看。** `top` 列列出激活最高的 token，`bottom` 列列出激活最低的 token。它不是模型 next-token 预测，也不是 vocab top-k。

**怎么交互。** 可以按 sample/layer 过滤，按 top/bottom 类型切换，或者在搜索框中输入 token 文本，比如 `attention` 或 `residual`。

**结合样例。** 如果某个 residual dimension 的 top token 总是和 `attention`、`heads` 相关，它可能对这段文本中的相关位置更敏感；但是否具备稳定语义还需要更多样本验证。

In [12]:
display(HTML(interactive_views['09_topk_input_token_browser'].html))

sample,layer,neuron,top,bottom
0,L0,resid_d0,".=0.2109; interactive=0.1836; views=-0.09277; ,=-0.1206",residual=-0.8359; and=-0.6562; Safe=-0.6289; attention=-0.4395
0,L0,resid_d1,predictions=0.9375; residual=0.9219; interactive=0.8516; dimensions=0.8516,",=-0.3301; ,=-0.2812; and=-0.209; .=-0.1953"
0,L0,resid_d2,Safe=0.5781; .=0.1582; model=0.08203; and=0.02942,"heads=-0.2949; attention=-0.1074; ,=-0.08643; Lens=-0.08398"
0,L0,resid_d3,attention=-0.3477; and=-0.4453; heads=-0.4863; predictions=-0.5039,",=-0.8906; .=-0.8789; views=-0.8203; Lens=-0.75"
0,L0,resid_d4,reveal=0.2969; interactive=0.2617; Lens=0.207; attention=0.1699,"Safe=-0.008911; ,=0.01172; .=0.02539; predictions=0.03735"
0,L0,resid_d5,".=0.1641; views=0.125; ,=0.05322; ,=0.04297",reveal=-0.332; Safe=-0.3066; residual=-0.291; interactive=-0.2061
0,L0,resid_d6,reveal=0.3516; heads=0.2354; model=0.2148; and=0.1953,residual=-0.2236; Safe=-0.2227; Lens=-0.2188; dimensions=-0.1855
0,L0,resid_d7,predictions=0.3086; interactive=-0.05762; reveal=-0.06543; attention=-0.09521,",=-1.812; ,=-1.656; and=-1.461; .=-1.164"
0,L1,resid_d0,".=0.4609; Safe=0.2676; ,=-0.1719; model=-0.2246",residual=-1.023; and=-1.016; attention=-0.9375; reveal=-0.918
0,L1,resid_d1,predictions=1.422; dimensions=1.375; interactive=1.312; residual=1.305,"Safe=-0.6406; ,=-0.1211; ,=-0.06689; and=-0.04199"


## 3.10 Feature-Linked Text Sample Browser

**展示对象。** 一个小型 demo text corpus，模拟“某个 feature 在训练/样本文本中最强响应”的浏览方式。

**怎么看。** 每行是某个 layer-feature 组合下的样本排名。`max_token` 是该样本里激活最大的 token，`max_value` 是对应值。

**怎么交互。** 用 layer/neuron 下拉框筛选 feature 位置，用搜索框查找 token 或样本内容。

**结合样例。** 例如 `unsafe` 排名靠前，表示 demo feature 在包含 `unsafe` 的样本上响应更高。这里展示交互能力，不声称这些是 Qwen 的真实训练数据检索结果。

In [13]:
display(HTML(interactive_views['10_feature_linked_text_sample_browser'].html))

layer,neuron,rank,sample,max_token,max_value
demo L0,demo feature 0,1,1,blocked,0.92
demo L0,demo feature 0,2,2,residual,0.77
demo L0,demo feature 0,3,0,safe,0.38
demo L0,demo feature 1,1,1,policy,0.88
demo L0,demo feature 1,2,2,residual,0.83
demo L0,demo feature 1,3,0,snippet,0.62
demo L1,demo feature 0,1,2,copy,0.81
demo L1,demo feature 0,2,1,policy,0.73
demo L1,demo feature 0,3,0,answer,0.61
demo L1,demo feature 1,1,2,copy,0.86


## 3.11 Qwen Next-Token Prediction Browser

**展示对象。** 真实 Qwen logits 派生出的每个位置 next-token prediction。

**怎么看。** `position` 表示从当前 token 预测下一个 token；表格列出 top-k 候选 token、logit、log_prob、prob，并标记真实 prompt 中的下一个 token。

**怎么交互。** 切换 position 可以查看不同上下文位置的预测；metric 下拉框可以在 logit/log_prob/prob 之间切换。

**结合样例。** 如果目标 token rank 很高，说明模型认为 prompt 中实际下一个 token 很自然；如果 rank 很低，说明该位置的续写较不符合模型偏好。

In [14]:
display(HTML(interactive_views['11_next_token_prediction_browser'].html))

rank,token,token_id,logit,target


## 3.12 Qwen Residual Dimension Cache Browser

**展示对象。** 真实 Qwen `resid_post` 前 16 个 residual dimensions 的 robust z-score。`resid_d0` 到 `resid_d15` 是隐藏维度编号，不是语义 feature。

**为什么用 robust z-score。** raw residual 里可能有极端 boundary-token 值，直接画会把色阶拉爆，让大部分格子接近白色。这里对每个维度做 median-centered / median-absolute-deviation 缩放，并 clamp 到 `[-4, 4]`，让结构可见。

**怎么交互。** activation 下拉框切换 layer；view 下拉框可以转置 token/dimension 视角；hover/click 单元格看具体 token-dimension 值。

**结合样例。** 行是 token，列是 residual dimension。某个红/蓝格子表示该 token 在某个隐藏维度上相对本 prompt 的其他 token 更高/更低。

In [15]:
display(HTML(interactive_views['12_residual_dimension_cache_browser'].html))

## 4. Execution metadata

下面的输出用于证明 notebook 是真实权重执行，并且交互部件数量和覆盖范围满足要求。

In [16]:
print("model_id:", MODEL_ID)
print("prompt:", PROMPT)
print("token_count:", len(viz_tokens))
print("real_attention_shape:", tuple(layer_head_patterns.shape))
print("real_residual_shape:", tuple(resid_by_layer.shape))
print("real_mlp_out_shape:", tuple(mlp_out_by_layer.shape))
print("real_mlp_post_shape:", tuple(mlp_post_by_layer.shape))
print("widget_count:", len(interactive_views))
print("component_category_count:", len(component_coverage))
print("interactive_operation_count:", len(interaction_coverage))
print("residual_dimension_labels:", resid_dim_labels_16)
print("mlp_output_dimension_labels:", mlp_out_dim_labels_8)
print("mlp_hidden_neuron_labels:", mlp_neuron_labels_8)
print("mlp_direction_neuron_labels:", mlp_direction_neuron_labels)
print("mlp_logit_contribution_targets:", list(mlp_logit_contributions.keys()))

model_id: Qwen/Qwen3-0.6B
prompt: SafeLens interactive views reveal attention heads, residual dimensions, and model predictions.
token_count: 15
real_attention_shape: (3, 4, 15, 15)
real_residual_shape: (3, 15, 1024)
real_mlp_out_shape: (3, 15, 1024)
real_mlp_post_shape: (3, 15, 3072)
widget_count: 12
component_category_count: 5
interactive_operation_count: 6
residual_dimension_labels: ['resid_d0', 'resid_d1', 'resid_d2', 'resid_d3', 'resid_d4', 'resid_d5', 'resid_d6', 'resid_d7', 'resid_d8', 'resid_d9', 'resid_d10', 'resid_d11', 'resid_d12', 'resid_d13', 'resid_d14', 'resid_d15']
mlp_output_dimension_labels: ['resid_d0', 'resid_d1', 'resid_d2', 'resid_d3', 'resid_d4', 'resid_d5', 'resid_d6', 'resid_d7']
mlp_hidden_neuron_labels: ['mlp_n0', 'mlp_n1', 'mlp_n2', 'mlp_n3', 'mlp_n4', 'mlp_n5', 'mlp_n6', 'mlp_n7']
mlp_direction_neuron_labels: ['mlp_n55', 'mlp_n128', 'mlp_n1489', 'mlp_n321', 'mlp_n46', 'mlp_n646', 'mlp_n0', 'mlp_n77']
mlp_logit_contribution_targets: ['actual next token', 'mo